<a href="https://colab.research.google.com/github/M4rck0/Datos_Masivos/blob/main/Tarea_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Librerías
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("SparkEnColab")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

sc = spark.sparkContext

In [3]:
# Datos
subreddit = "technology"
ruta_parquet = f"/content/reddit_spark/{subreddit}"

df = (spark.read
      .option("recursiveFileLookup", "true")
      .parquet(ruta_parquet)
      .select("created_utc", "author", "score", "body")
     )

# Quitar eliminados
df = df.filter(~((F.col("author") == "[deleted]") | (F.col("body") == "[deleted]") | (F.col("body") == "[removed]")))

# Quitar nulos
df = df.filter(F.col("score").isNotNull())

print("Filas en parquet:", df.count())
df.show(5, truncate=50)

Filas en parquet: 180078
+-----------+-------------+-----+--------------------------------------------------+
|created_utc|       author|score|                                              body|
+-----------+-------------+-----+--------------------------------------------------+
| 1432313203|      AbeRego|    3|   I read this in the cliché teen Simpson's voice.|
| 1432313210|   hefnetefne|    1|How about a law that says you can sue corporati...|
| 1432313243|newloginisnew|   23|Please share 100% of your browsing history as a...|
| 1432313244|       NeonHD|    1|My post wasn't supposed to be a question, it wa...|
| 1432313246|        Zoura|   47|As a former BBV employee I would just like to s...|
+-----------+-------------+-----+--------------------------------------------------+
only showing top 5 rows



In [4]:
# Convertir a RDD
rdd_filas = df.rdd
type(rdd_filas.first())


pyspark.sql.types.Row

In [5]:
rdd_filas.take(2) # Ver 2 filas

[Row(created_utc=1432313203, author='AbeRego', score=3, body="I read this in the cliché teen Simpson's voice."),
 Row(created_utc=1432313210, author='hefnetefne', score=1, body="How about a law that says you can sue corporations, instead of proclaiming corporations people so you can sue them? \n\nThat ever cross anyone's mind?")]

In [6]:
def estadistica(a, b):
    # Recibe dos tuplas (n, suma, suma cuadrada, minimo, maximo) y devuelve tupla
    n1, s1, ss1, mn1, mx1 = a
    n2, s2, ss2, mn2, mx2 = b
    return (n1+n2, s1+s2, ss1+ss2, min(mn1, mn2), max(mx1, mx2))

In [7]:
scores = rdd_filas.map(lambda r: r.score)
operaciones = scores.map(lambda v: (1, v, v*v, v, v))
n, suma, suma_cuadrados, minimo, maximo = operaciones.reduce(estadistica)
media = suma / n
varianza = (suma_cuadrados / n) - (media**2) # Formula simplificada

In [8]:
print("Longitud:", n)
print("Media:", media)
print("Mínimo:", minimo)
print("Máximo:", maximo)
print("Varianza:", varianza)

Longitud: 180078
Media: 15.149379713235376
Mínimo: -366
Máximo: 23281
Varianza: 30064.900574671607


In [9]:
scores_altos = scores.filter(lambda s: s >= 300)
print("Scores >= 300:", scores_altos.count())
print("Ejemplo scores >= 300:", scores_altos.take(5))

Scores >= 300: 1317
Ejemplo scores >= 300: [2012, 521, 589, 471, 1160]
